# Figure 4 | Striosome-matrix changes in HD

191-gene spatial maps, S/M score differences, DC1 score profiles, heatmaps, and CV.

Check input/output paths and settings before running each workflow. Run its cells in order with a fresh kernel. See README.md for required inputs, panel coverage, and configuration differences.

| Panels | Workflow |
|---|---|
| B | Tshz1/Id4 tissue maps |
| C | S-score minus M-score tissue maps |
| D | Four DC1 delta-score profiles |
| E | DC1 gene heatmaps |
| F | CV from verified linear expression |
| A | Schematic |


## Panel B | Tshz1 and Id4 tissue maps

Expression is read from `norm_1e4` after matching cell and gene IDs. The manuscript legend specifies raw counts; the expression scale and legend require reconciliation.


In [ ]:
# Imports and settings.
from pathlib import Path
import gc
import json
import logging
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm, colors
from scipy import sparse
import scanpy as sc

DATA_ROOT = Path(r"F:\BASSFISH_HD\Source_data")
TARGET_FILES = {
    "D1": DATA_ROOT / "d1_sm_data_final.h5ad",
    "D2": DATA_ROOT / "d2_sm_data_final.h5ad",
}
EXPRESSION_FILE = DATA_ROOT / "integrated_5_str_with_axis_xy.h5ad"
EXPRESSION_LAYER = "norm_1e4"
OUT_ROOT = (Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\sm")
            / "single_gene")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

GENES = ["Id4","Tshz1"]
CONDITIONS = ["WT", "Heter", "Homo"]
CONDITION_ALIASES = {"HD": "Heter"}
SAMPLE_COL, CONDITION_COL, XCOL, YCOL = "sample", "condition", "x", "y"


MIN_OVERALL_MATCH_FRACTION = 0.95
MIN_SAMPLE_MATCH_FRACTION = 0.90
STRICT_ANNOTATION_CONSISTENCY = True


CMAP = "viridis"
VMIN = 0.0
VMAX = None
VMAX_REFERENCE = "all"
VMAX_POSITIVE_PERCENTILE = 99.5


MAX_COLUMNS = 4
PANEL_W_MM = 57
PANEL_H_MM = 68
POINT_SIZE = 0.55
POINT_ALPHA = 0.90
BACKGROUND_POINT_COLOR = "#DDDDDD"
BACKGROUND_ALPHA = 0.72
DPI = 600
SHOW_FIGURE = False

MICRONS_PER_UNIT = 0.16
SCALE_BAR_UM = 500
SHOW_SCALEBAR_EVERY_PANEL = True
INVERT_Y = True

plt.rcParams.update({
    "font.family": "Arial", "font.size": 7,
    "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "figure.max_open_warning": 10,
})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)


In [ ]:
def normalize_conditions(series):
    return series.astype("string").str.strip().replace(CONDITION_ALIASES)


def read_expression_reference():
    if not EXPRESSION_FILE.is_file():
        raise FileNotFoundError(f"Expression source file not found: {EXPRESSION_FILE}")
    src = sc.read_h5ad(EXPRESSION_FILE)
    try:
        if not src.obs_names.is_unique or not src.var_names.is_unique:
            raise ValueError("Expression source has duplicate cell IDs or gene names; alignment is ambiguous")
        if EXPRESSION_LAYER not in src.layers:
            raise KeyError(f"Expression source is missing {EXPRESSION_LAYER!r}; available layers={list(src.layers.keys())}")
        expr = src.layers[EXPRESSION_LAYER]
        if expr.shape != (src.n_obs, src.n_vars):
            raise ValueError(f"Expression matrix shape does not match h5ad: {expr.shape} != {src.shape}")
        for gene in GENES:
            if gene not in src.var_names:
                raise KeyError(f"Gene {gene!r} is absent from the source; check its spelling and the 191-gene panel")
        metadata_cols = [col for col in (SAMPLE_COL, CONDITION_COL) if col in src.obs.columns]
        result = dict(
            matrix=expr,
            cell_ids=src.obs_names.copy(),
            gene_ids=src.var_names.copy(),
            meta=src.obs[metadata_cols].copy(),
        )
        print(f"[Expression source] {src.n_obs:,} cells x {src.n_vars} genes; layer={EXPRESSION_LAYER}")
        return result
    finally:

        del src
        gc.collect()


def target_metadata(name, path, ref):
    if not path.is_file():
        raise FileNotFoundError(f"{name} file not found: {path}")
    adata = sc.read_h5ad(path)
    try:
        needed = [SAMPLE_COL, CONDITION_COL, XCOL, YCOL]
        absent = [col for col in needed if col not in adata.obs.columns]
        if absent:
            raise KeyError(f"{name}: obs is missing fields {absent}; available: {list(adata.obs.columns)}")
        if not adata.obs_names.is_unique:
            raise ValueError(f"{name}: obs_names contains duplicate cell IDs; automatic deduplication is disabled")
        data = pd.DataFrame({
            "cell_id": adata.obs_names.astype(str),
            "msn_type": name,
            "sample": adata.obs[SAMPLE_COL].astype("string").str.strip().to_numpy(),
            "condition": normalize_conditions(adata.obs[CONDITION_COL]).to_numpy(),
            "x": pd.to_numeric(adata.obs[XCOL], errors="coerce").to_numpy(dtype=float),
            "y": pd.to_numeric(adata.obs[YCOL], errors="coerce").to_numpy(dtype=float),
        }).reset_index(drop=True)
    finally:
        del adata
        gc.collect()

    if data["sample"].isna().any() or data["condition"].isna().any():
        raise ValueError(f"{name}: sample/condition contains missing values; grouping is undefined")
    invalid_conditions = sorted(set(data["condition"]) - set(CONDITIONS))
    if invalid_conditions:
        raise ValueError(f"{name}: unrecognized condition={invalid_conditions}; check CONDITION_ALIASES")

    data["source_row"] = ref["cell_ids"].get_indexer(data["cell_id"])
    data["matched"] = data["source_row"] >= 0
    data["finite_xy"] = np.isfinite(data["x"]) & np.isfinite(data["y"])

    grp = (data.groupby(["condition", "sample"], dropna=False, observed=True)
           .agg(n_target=("cell_id", "size"),
                n_matched=("matched", "sum"),
                n_finite_xy=("finite_xy", "sum"))
           .reset_index())
    grp["msn_type"] = name
    grp["n_unmatched"] = grp["n_target"] - grp["n_matched"]
    grp["match_fraction"] = grp["n_matched"] / grp["n_target"]
    match_fraction = float(data["matched"].mean())
    print(f"[{name}] source cell ID match: {data['matched'].sum():,}/{len(data):,} "
          f"({match_fraction:.2%}); invalid xy={(~data['finite_xy']).sum():,}")
    if not bool(data["matched"].any()):
        raise ValueError(f"{name}: no cell IDs match the expression source; confirm that both files use the same obs_names")
    return data, grp


def load_and_align_targets(ref):
    tables, group_tables = [], []
    for name, path in TARGET_FILES.items():
        t, gr = target_metadata(name, path, ref)
        tables.append(t)
        group_tables.append(gr)
    all_target = pd.concat(tables, ignore_index=True)
    all_groups = pd.concat(group_tables, ignore_index=True)
    all_groups.to_csv(OUT_ROOT / "00_cell_match_coverage_by_sample_and_type.csv", index=False)
    all_target.loc[~all_target["matched"], ["cell_id", "msn_type", "condition", "sample"]].to_csv(
        OUT_ROOT / "00_unmatched_target_cell_ids.csv", index=False)

    dup = all_target[all_target["cell_id"].duplicated(keep=False)]
    if not dup.empty:
        dup[["cell_id", "msn_type", "condition", "sample"]].to_csv(
            OUT_ROOT / "00_D1_D2_duplicate_ids_ERROR.csv", index=False)
        raise ValueError(f"D1/D2 contain {dup['cell_id'].nunique()} duplicate cell IDs; duplicate counting is not allowed. See the QC CSV")

    for name in TARGET_FILES:
        t = all_target[all_target["msn_type"] == name]
        pct = float(t["matched"].mean())
        if pct < MIN_OVERALL_MATCH_FRACTION:
            raise ValueError(f"{name} ID match rate {pct:.2%} < {MIN_OVERALL_MATCH_FRACTION:.0%}; see the matching QC CSV")
    small = all_groups[all_groups["match_fraction"] < MIN_SAMPLE_MATCH_FRACTION]
    if not small.empty:
        raise ValueError(f"{len(small)} sample-by-MSN-type groups fall below the match-rate threshold "
                         f"{MIN_SAMPLE_MATCH_FRACTION:.0%}; see 00_cell_match_coverage_by_sample_and_type.csv")

    matched = all_target[all_target["matched"]].copy().reset_index(drop=True)
    if STRICT_ANNOTATION_CONSISTENCY:
        meta = ref["meta"]

        source_meta = meta.reindex(matched["cell_id"])
        mismatch = np.zeros(len(matched), dtype=bool)
        details = matched[["cell_id", "msn_type", "sample", "condition"]].copy()
        if SAMPLE_COL in meta.columns:
            ss = source_meta[SAMPLE_COL].astype("string").str.strip().to_numpy()
            mismatch |= (ss != matched["sample"].to_numpy())
            details["source_sample"] = ss
        if CONDITION_COL in meta.columns:
            scnd = normalize_conditions(source_meta[CONDITION_COL]).to_numpy()
            mismatch |= (scnd != matched["condition"].to_numpy())
            details["source_condition"] = scnd
        if mismatch.any():
            details.loc[mismatch].to_csv(OUT_ROOT / "00_source_target_annotation_mismatch_ERROR.csv", index=False)
            raise ValueError(f"Source/target sample or condition differs for the same cell ID "
                             f"({mismatch.sum()} cells); see the QC CSV")


    count_cond = matched.groupby("sample", observed=True)["condition"].nunique()
    if (count_cond > 1).any():
        bad = count_cond[count_cond > 1].index.tolist()
        raise ValueError(f"The same sample name maps to multiple conditions: {bad}; resolve the sample naming first")
    matched = matched[matched["finite_xy"]].copy().reset_index(drop=True)
    if matched.empty:
        raise ValueError("All matched cells lack valid x/y coordinates")
    audit = {
        "target_files": {key: str(path) for key, path in TARGET_FILES.items()},
        "expression_file": str(EXPRESSION_FILE),
        "expression_layer": EXPRESSION_LAYER,
        "total_target_cells": int(len(all_target)),
        "n_matched_cells": int(all_target["matched"].sum()),
        "n_matched_finite_xy_cells_used": int(len(matched)),
        "n_unmatched": int((~all_target["matched"]).sum()),
        "n_duplicate_d1_d2": 0,
        "condition_aliases": CONDITION_ALIASES,
    }
    (OUT_ROOT / "00_alignment_audit.json").write_text(
        json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    return matched, all_groups


def extract_aligned_gene(ref, data, gene):
    pos = ref["gene_ids"].get_loc(gene)
    rows = data["source_row"].to_numpy(dtype=np.int64)
    result = ref["matrix"][rows, pos]
    if sparse.issparse(result):
        result = result.toarray()
    result = np.asarray(result, dtype=float).reshape(-1)
    if len(result) != len(data):
        raise AssertionError(f"{gene}: row count mismatch {len(result)} vs {len(data)}")
    if not np.all(np.isfinite(result)):
        raise ValueError(f"{gene}: norm_1e4 contains NaN/inf and cannot be plotted reliably")
    if np.any(result < -1e-8):
        raise ValueError(f"{gene}: negative expression values detected; scaled X cannot be used as norm_1e4")
    return np.maximum(result, 0)


In [ ]:
def add_scalebar(ax, xmin, xmax, ymin, ymax):
    if not SHOW_SCALEBAR_EVERY_PANEL:
        return
    dx, dy = xmax - xmin, ymax - ymin
    length_coord = SCALE_BAR_UM / MICRONS_PER_UNIT
    if length_coord > dx * 0.80:
        print(f"[WARN] The {SCALE_BAR_UM} micrometer scale bar exceeds 80% of the global x span and was omitted; check coordinate units")
        return
    x0 = xmin + .08 * dx

    y0 = (ymax - .08 * dy) if INVERT_Y else (ymin + .08 * dy)
    ax.plot([x0, x0 + length_coord], [y0, y0],
            color="black", lw=1.15, solid_capstyle="butt", zorder=6)
    text_y = y0 - .025 * dy if INVERT_Y else y0 + .025 * dy
    ax.text(x0 + length_coord / 2, text_y, f"{SCALE_BAR_UM} µm",
            ha="center", va="bottom" if INVERT_Y else "top",
            fontsize=7, color="black", zorder=7,
            bbox=dict(facecolor="white", edgecolor="none", alpha=.60, pad=.5))


def gene_color_scale(values, conditions):
    if VMAX is not None:
        vmax = float(VMAX)
    else:
        if VMAX_REFERENCE not in ("WT", "all"):
            raise ValueError("VMAX_REFERENCE must be 'WT' or 'all'")
        refmask = conditions == "WT" if VMAX_REFERENCE == "WT" else np.ones(len(values), dtype=bool)
        positive = values[refmask & (values > 0)]
        if len(positive) == 0:
            raise ValueError("The automatic color-scale reference has no positive cells; set VMAX or use VMAX_REFERENCE='all'")
        vmax = float(np.percentile(positive, VMAX_POSITIVE_PERCENTILE))
    if not np.isfinite(vmax) or vmax <= VMIN:
        raise ValueError(f"Invalid color limits VMIN={VMIN}, VMAX={vmax}")
    return vmax


def sorted_panels(df):

    rows = []
    for c in CONDITIONS:
        smps = sorted(pd.unique(df.loc[df["condition"] == c, "sample"]))
        for s in smps:
            rows.append((c, str(s)))
    if not rows:
        raise ValueError("No WT/Heter/Homo samples are available for display")
    return rows


def render_gene(gene, data, gene_values):
    cond = data["condition"].astype(str).to_numpy()
    sample = data["sample"].astype(str).to_numpy()
    typ = data["msn_type"].astype(str).to_numpy()
    x = data["x"].to_numpy(dtype=float)
    y = data["y"].to_numpy(dtype=float)
    vmax = gene_color_scale(gene_values, cond)
    panels = sorted_panels(data)
    print(f"[{gene}] {len(panels)} samples; shared colorbar: {VMIN:g} - {vmax:.4g}")


    xmin, xmax = float(np.min(x)), float(np.max(x))
    ymin, ymax = float(np.min(y)), float(np.max(y))
    if not (xmax > xmin and ymax > ymin):
        raise ValueError("The x or y range is zero for all samples; a two-dimensional spatial plot cannot be drawn")
    padx, pady = .02 * (xmax - xmin), .02 * (ymax - ymin)
    xmin, xmax = xmin - padx, xmax + padx
    ymin, ymax = ymin - pady, ymax + pady

    ncols = min(MAX_COLUMNS, len(panels))
    nrows = math.ceil(len(panels) / ncols)
    fw_mm = 14 + ncols * PANEL_W_MM + 4 * (ncols - 1) + 17
    fh_mm = 14 + nrows * PANEL_H_MM + 6 * (nrows - 1) + 11
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(fw_mm / 25.4, fh_mm / 25.4),
                             squeeze=False, facecolor="white")
    fig.subplots_adjust(left=12/fw_mm, right=(fw_mm-19)/fw_mm,
                        bottom=9/fh_mm, top=(fh_mm-8)/fh_mm,
                        wspace=.14, hspace=.20)
    norm = colors.Normalize(vmin=VMIN, vmax=vmax, clip=True)
    cmap_obj = plt.get_cmap(CMAP)
    records = []
    try:
        for i, (c, s) in enumerate(panels):
            ax = axes.flat[i]
            idx = np.flatnonzero((cond == c) & (sample == s))
            if len(idx) == 0:
                continue
            positive = idx[gene_values[idx] > 0]

            ax.scatter(x[idx], y[idx], s=POINT_SIZE,
                       color=BACKGROUND_POINT_COLOR, alpha=BACKGROUND_ALPHA,
                       linewidths=0, rasterized=False, zorder=1)
            if len(positive):
                positive = positive[np.argsort(gene_values[positive], kind="stable")]
                ax.scatter(x[positive], y[positive],
                           c=gene_values[positive], cmap=cmap_obj, norm=norm,
                           s=POINT_SIZE, alpha=POINT_ALPHA,
                           linewidths=0, rasterized=False, zorder=2)
            ax.set_xlim(xmin, xmax)
            ax.set_ylim(ymax, ymin) if INVERT_Y else ax.set_ylim(ymin, ymax)
            ax.set_aspect("equal", adjustable="box")
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_facecolor("white")
            ax.set_title(f"{c} | {s}", fontsize=7, color="black", pad=2)
            for spine in ax.spines.values():
                spine.set_linewidth(.45)
            add_scalebar(ax, xmin, xmax, ymin, ymax)
            records.append({
                "gene": gene, "condition": c, "sample": s,
                "n_MSN": int(len(idx)),
                "n_D1": int(np.sum(typ[idx] == "D1")),
                "n_D2": int(np.sum(typ[idx] == "D2")),
                "n_expressing": int(len(positive)),
                "fraction_expressing": float(len(positive) / len(idx)),
                "mean_norm_1e4": float(np.mean(gene_values[idx])),
                "median_norm_1e4": float(np.median(gene_values[idx])),
                "p99_norm_1e4": float(np.percentile(gene_values[idx], 99)),
            })
        for ax in list(axes.flat)[len(panels):]:
            ax.set_visible(False)
        cax = fig.add_axes([(fw_mm-13)/fw_mm,
                            max(0.20, .33 if nrows == 1 else .36),
                            2.3/fw_mm,
                            .50 if nrows == 1 else .28])
        cb = fig.colorbar(cm.ScalarMappable(norm=norm, cmap=cmap_obj), cax=cax)
        cb.set_label(f"{gene}  ({EXPRESSION_LAYER})", fontsize=7)
        cb.ax.tick_params(labelsize=6, width=.5, length=2, pad=1)
        cb.outline.set_linewidth(.4)
        pdf = OUT_ROOT / f"{gene}_spatial_merged_D1_D2_MSN_all_samples.pdf"
        fig.savefig(pdf, dpi=DPI, facecolor="white", transparent=False,
                    bbox_inches="tight", pad_inches=.02)
        if SHOW_FIGURE:
            plt.show()
    finally:
        plt.close(fig)
        del fig, axes
        gc.collect()

    stats_file = OUT_ROOT / f"{gene}_sample_MSN_expression_QC.csv"
    pd.DataFrame(records).to_csv(stats_file, index=False)
    params = {
        "gene": gene, "expression_layer": EXPRESSION_LAYER,
        "expression_file": str(EXPRESSION_FILE),
        "n_total_MSN": int(len(data)),
        "n_D1": int(np.sum(typ == "D1")),
        "n_D2": int(np.sum(typ == "D2")),
        "n_samples": len(panels),
        "vmin": float(VMIN), "vmax": float(vmax),
        "vmax_reference": VMAX_REFERENCE if VMAX is None else "manual",
        "vmax_positive_percentile": VMAX_POSITIVE_PERCENTILE if VMAX is None else None,
        "coordinate_um_per_unit": MICRONS_PER_UNIT,
        "scale_bar_um": SCALE_BAR_UM,
        "xmin": xmin, "xmax": xmax, "ymin": ymin, "ymax": ymax,
        "dpi": DPI, "scatter_is_vector": True,
    }
    (OUT_ROOT / f"{gene}_plot_parameters.json").write_text(
        json.dumps(params, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[saved] {pdf}")
    return params


In [ ]:
def run_all():
    ref = read_expression_reference()
    try:
        data, match_qc = load_and_align_targets(ref)
        print(f"[Plotting] {len(data):,} MSN: "
              f"D1={(data['msn_type']=='D1').sum():,}, D2={(data['msn_type']=='D2').sum():,}")
        print(f"[sample] {len(sorted_panels(data))} samples, ordered WT -> Heter -> Homo")
        manifests = []
        for gene in dict.fromkeys(GENES):
            val = extract_aligned_gene(ref, data, gene)
            result = render_gene(gene, data, val)
            manifests.append(result)
            del val
            gc.collect()
        pd.DataFrame(manifests).to_csv(OUT_ROOT / "_gene_spatial_PDF_manifest.csv", index=False)
        print("\nCompleted:", OUT_ROOT)
        return manifests
    finally:
        del ref
        gc.collect()

RESULTS = run_all()


## Panel C | S-score minus M-score

Defaults are `VALUE_MODE="raw"`, `CMAP="viridis"`, and `RAW_VMIN/RAW_VMAX=-3/3`. In raw mode, the separate `VMIN/VMAX=-0.5/0.5` settings are inactive. The manuscript legend specifies -0.5 to 0.5.


In [ ]:
# Imports and settings.
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt

OUT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\sm\s_m_spatial")
OUT.mkdir(parents=True, exist_ok=True)
FILES = {
    "D1": Path(r"F:\BASSFISH_HD\Source_data\d1_sm_data_final.h5ad"),
    "D2": Path(r"F:\BASSFISH_HD\Source_data\d2_sm_data_final.h5ad"),
}
ROWS=["D1","D2"]; COND_ORDER=["WT","Heter","Homo"]; SAMPLE_COL="sample"; COND_COL="condition"; XCOL="x"; YCOL="y"; S_COL="S_score"; M_COL="M_score"


VALUE_MODE="raw"


CMAP = "viridis"
RAW_VMIN, RAW_VMAX = -3.0, 3.0

VMIN, VMAX = -0.5, 0.5
POINT_SIZE=1; ALPHA=0.8; DPI=600

MICRONS_PER_UNIT=0.16; SCALE_BAR_UM=500; SHOW_SCALEBAR_EVERY_PANEL=True
PANEL_W=110; PANEL_H=140; GAP=2; GROUP_GAP=5; LEFT=14; RIGHT=5; BOTTOM=12; TOP=10; CB_W=3; CB_GAP=4

plt.rcParams.update({"font.family":"Arial","font.size":7,"axes.labelsize":7,"axes.titlesize":7,
                     "xtick.labelsize":7,"ytick.labelsize":7,"pdf.fonttype":42,"ps.fonttype":42})


In [ ]:
def add_scalebar(ax,xmin,xmax,ymin,ymax,um_per_unit=.16,bar_um=500):
    xr=xmax-xmin; yr=ymax-ymin; L=bar_um/um_per_unit
    x0=xmin+0.08*xr; y0=ymax-0.08*yr; x1=x0+L
    if x1>xmax-0.05*xr: x1=xmax-0.05*xr; x0=x1-L
    ax.plot([x0,x1],[y0,y0],lw=1.2,color="black",solid_capstyle="butt",zorder=5)
    ax.text((x0+x1)/2,y0-0.02*yr,f"{bar_um} μm",ha="center",va="bottom",fontsize=7,color="black")

def load_all():
    data={}; pairs=[]; gx=[]; gy=[]; all_diff=[]
    for ct in ROWS:
        f=FILES[ct]
        if not f.exists(): raise FileNotFoundError(f)
        ad=sc.read_h5ad(f)
        for col in [SAMPLE_COL,COND_COL,XCOL,YCOL,S_COL,M_COL]:
            if col not in ad.obs: raise KeyError(f"{ct}: obs is missing {col}")
        x=pd.to_numeric(ad.obs[XCOL],errors="coerce").to_numpy(float)
        y=pd.to_numeric(ad.obs[YCOL],errors="coerce").to_numpy(float)
        s=pd.to_numeric(ad.obs[S_COL],errors="coerce").to_numpy(float)
        m=pd.to_numeric(ad.obs[M_COL],errors="coerce").to_numpy(float)
        diff=s-m
        cond=ad.obs[COND_COL].astype(str).to_numpy(); sample=ad.obs[SAMPLE_COL].astype(str).to_numpy()
        good=np.isfinite(x)&np.isfinite(y)&np.isfinite(diff)
        data[ct]={"adata":ad,"x":x,"y":y,"s":s,"m":m,"diff":diff,"cond":cond,"sample":sample,"good":good}
        gx.append(x[good]); gy.append(y[good]); all_diff.append(diff[good])
        for c in COND_ORDER:
            for smp in sorted(pd.unique(sample[good&(cond==c)])): pairs.append((c,smp))
        print(ct,ad)

    seen=set(); panels=[]
    for c,smp in pairs:
        if (c,smp) not in seen: panels.append((c,smp)); seen.add((c,smp))


    for ct in ROWS:
        available = set(zip(data[ct]["cond"][data[ct]["good"]],
                            data[ct]["sample"][data[ct]["good"]]))
        missing = [pair for pair in panels if pair not in available]
        if missing:
            print(f"[Warning] {ct} is missing {len(missing)} sample/condition combinations: {missing}")

    gx=np.concatenate(gx); gy=np.concatenate(gy); gd=np.concatenate(all_diff)
    xmin,xmax=np.nanmin(gx),np.nanmax(gx); ymin,ymax=np.nanmin(gy),np.nanmax(gy)
    padx=.02*(xmax-xmin) if xmax>xmin else 1; pady=.02*(ymax-ymin) if ymax>ymin else 1
    extent=(xmin-padx,xmax+padx,ymin-pady,ymax+pady)

    if VALUE_MODE=="global_maxabs_1":
        scale=np.nanmax(np.abs(gd))
        if not np.isfinite(scale) or scale<=0: scale=1.0
        vmin,vmax=VMIN,VMAX
    elif VALUE_MODE=="raw":
        lo=np.nanmin(gd) if RAW_VMIN is None else RAW_VMIN; hi=np.nanmax(gd) if RAW_VMAX is None else RAW_VMAX
        m=max(abs(lo),abs(hi)); scale=None; vmin,vmax=-m,m
    elif VALUE_MODE=="global_zscore":
        mu=np.nanmean(gd); sd=np.nanstd(gd,ddof=0); scale=(mu,sd)
        zz=(gd-mu)/sd if sd>0 else gd*0
        m=np.nanmax(np.abs(zz)) if np.isfinite(zz).any() else 1
        vmin,vmax=-m,m
    else:
        raise ValueError("VALUE_MODE must be global_maxabs_1 / raw / global_zscore")

    return data,panels,extent,scale,vmin,vmax

def transform(diff,scale):
    diff=np.asarray(diff,float)
    if VALUE_MODE=="global_maxabs_1": return diff/scale
    if VALUE_MODE=="raw": return diff
    mu,sd=scale
    if sd<=0:
        out=np.zeros_like(diff,float); out[~np.isfinite(diff)]=np.nan; return out
    return (diff-mu)/sd

def prepare_values(data,scale):
    qc_rows=[]; cell_rows=[]; plot_values={}
    for ct in ROWS:
        d=data[ct]; plot=np.full_like(d["diff"],np.nan,float)
        plot[d["good"]]=transform(d["diff"][d["good"]],scale)
        plot_values[ct]=plot
        for c in COND_ORDER:
            for smp in sorted(pd.unique(d["sample"][d["good"]&(d["cond"]==c)])):
                idx=np.where(d["good"]&(d["cond"]==c)&(d["sample"]==smp))[0]; vv=d["diff"][idx]; pv=plot[idx]
                qc_rows.append({"celltype":ct,"condition":c,"sample":smp,"n_cells":len(idx),
                                "diff_min":np.nanmin(vv) if len(idx) else np.nan,"diff_max":np.nanmax(vv) if len(idx) else np.nan,
                                "diff_mean":np.nanmean(vv) if len(idx) else np.nan,"diff_median":np.nanmedian(vv) if len(idx) else np.nan,
                                "plot_min":np.nanmin(pv) if len(idx) else np.nan,"plot_max":np.nanmax(pv) if len(idx) else np.nan})
        cell_rows.append(pd.DataFrame({"celltype":ct,"cell":d["adata"].obs_names.astype(str),"condition":d["cond"],"sample":d["sample"],
                                       "x":d["x"],"y":d["y"],"S_score":d["s"],"M_score":d["m"],"SM_diff_raw":d["diff"],"plot_value":plot}))
    pd.DataFrame(qc_rows).to_csv(OUT/"D1_D2_SMdiff_spatial_globalnorm_sample_QC.csv",index=False)
    pd.concat(cell_rows,ignore_index=True).to_csv(OUT/"D1_D2_SMdiff_spatial_globalnorm_plot_values.csv",index=False)
    return plot_values

def plot_all(data,panels,extent,scale,vmin,vmax):
    plot_values=prepare_values(data,scale); xmin,xmax,ymin,ymax=extent
    lefts=[]; xpos=LEFT
    for i,(c,smp) in enumerate(panels):
        if i>0 and c!=panels[i-1][0]: xpos+=GROUP_GAP-GAP
        lefts.append(xpos); xpos+=PANEL_W+GAP
    right_grid=lefts[-1]+PANEL_W; fw=right_grid+CB_GAP+CB_W+RIGHT; fh=BOTTOM+2*PANEL_H+GAP+TOP

    fig=plt.figure(figsize=(fw/25.4,fh/25.4)); last=None; row_bottom={"D2":BOTTOM,"D1":BOTTOM+PANEL_H+GAP}
    for ct in ROWS:
        d=data[ct]
        for i,(c,smp) in enumerate(panels):
            ax=fig.add_axes([lefts[i]/fw,row_bottom[ct]/fh,PANEL_W/fw,PANEL_H/fh])
            idx=np.where(d["good"]&(d["cond"]==c)&(d["sample"]==smp)&np.isfinite(plot_values[ct]))[0]
            if len(idx):
                last=ax.scatter(d["x"][idx],d["y"][idx],c=plot_values[ct][idx],s=POINT_SIZE,cmap=CMAP,vmin=vmin,vmax=vmax,
                                linewidths=0,alpha=ALPHA,rasterized=True)
            ax.set_xlim(xmin,xmax); ax.set_ylim(ymax,ymin); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
            if ct=="D1": ax.set_title(f"{smp}\n{c}",pad=2)
            for sp in ax.spines.values(): sp.set_linewidth(.5)
            if SHOW_SCALEBAR_EVERY_PANEL: add_scalebar(ax,xmin,xmax,ymin,ymax,MICRONS_PER_UNIT,SCALE_BAR_UM)

    for i in range(1,len(panels)):
        if panels[i][0]!=panels[i-1][0]:
            sep=(lefts[i-1]+PANEL_W+lefts[i])/2
            fig.add_artist(plt.Line2D([sep/fw,sep/fw],[BOTTOM/fh,(BOTTOM+2*PANEL_H+GAP)/fh],transform=fig.transFigure,color="0.55",lw=.6,ls="--"))

    fig.text((LEFT-8)/fw,(row_bottom["D1"]+PANEL_H/2)/fh,"D1",ha="center",va="center",rotation=90,fontsize=8)
    fig.text((LEFT-8)/fw,(row_bottom["D2"]+PANEL_H/2)/fh,"D2",ha="center",va="center",rotation=90,fontsize=8)

    cax=fig.add_axes([(right_grid+CB_GAP)/fw,BOTTOM/fh,CB_W/fw,(2*PANEL_H+GAP)/fh]); cb=fig.colorbar(last,cax=cax)
    lab={"global_maxabs_1":"S_score - M_score (global maxabs = 1)","raw":"S_score - M_score","global_zscore":"S_score - M_score (global z-score)"}[VALUE_MODE]
    cb.set_label(lab); cb.ax.tick_params(width=.5,length=2,pad=1); cb.outline.set_linewidth(.5)

    pdf = OUT / f"D1_D2_SMdiff_spatial_all_samples_same_extent_{VALUE_MODE}_{CMAP}_600dpi.pdf"
    try:
        fig.savefig(pdf, dpi=DPI, bbox_inches="tight", pad_inches=.02,
                    transparent=True, format="pdf")
    finally:
        plt.close(fig)
    print("Saved PDF:", pdf)


In [ ]:
data,panels,extent,scale,vmin,vmax=load_all()
pd.DataFrame({"condition":[c for c,s in panels],"sample":[s for c,s in panels]}).to_csv(OUT/"D1_D2_SMdiff_spatial_globalnorm_panel_order.csv",index=False)
pd.DataFrame([{"value_mode":VALUE_MODE,"scale":str(scale),"vmin":vmin,"vmax":vmax,"microns_per_unit":MICRONS_PER_UNIT,"scale_bar_um":SCALE_BAR_UM}]).to_csv(
    OUT/"D1_D2_SMdiff_spatial_globalnorm_parameters.csv",index=False)
plot_all(data,panels,extent,scale,vmin,vmax)
print("\nDone:",OUT)


qc = pd.read_csv(OUT / "D1_D2_SMdiff_spatial_globalnorm_sample_QC.csv")
summary = (qc.groupby(["celltype", "condition"], observed=True)
           .agg(n_animals=("diff_mean", "count"),
                mean_of_animal_means=("diff_mean", "mean"),
                sd_of_animal_means=("diff_mean", "std"))
           .reset_index())
summary["sem_of_animal_means"] = summary["sd_of_animal_means"] / np.sqrt(summary["n_animals"])
summary.to_csv(OUT / "D1_D2_SMdiff_animal_level_summary.csv", index=False)
print("Saved animal-level descriptive summary:", OUT / "D1_D2_SMdiff_animal_level_summary.csv")


## Panel D | DC1 delta-score profiles

Uses WT-reference deciles and the per-animal baseline from bins 1-3.


In [ ]:
# Imports and settings.
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator


OUT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\sm\sm_dc")
OUT.mkdir(parents=True, exist_ok=True)

FILES = {
    "D1": Path(r"F:\BASSFISH_HD\Source_data\d1_sm_data_final.h5ad"),
    "D2": Path(r"F:\BASSFISH_HD\Source_data\d2_sm_data_final.h5ad"),
}
CELLTYPES = ["D1", "D2"]
CONDITIONS = ["WT", "Heter", "Homo"]
CONDITION_ALIASES = {"HD": "Heter"}

DC1_COL = "dc_component1"
S_COL, M_COL = "S_score", "M_score"
SAMPLE_COL, CONDITION_COL = "sample", "condition"


N_BINS = 10
BINNING_MODE = "WT_reference"
REVERSE_DC1 = {"D1": False, "D2": False}
BASELINE_BINS = [1, 2, 3]
MIN_CELLS_PER_BIN = 5
MIN_BASELINE_VALID_BINS = 2


SHOW_SEM = True
DPI = 600
COLORS = {"WT": "#4cd964", "Heter": "#ff9500", "Homo": "#ff3b30"}


PLOT_CFG = {
    ("D1", "delta_S"): {
        "ylim": (-0.3, 1.6),
        "ytick_step": 0.5,
        "ax_w_mm": 25, "ax_h_mm": 30,
        "left_mm": 12, "right_mm": 20, "bottom_mm": 12, "top_mm": 7,
        "ylabel": "ΔS score",
        "title": "D1: ΔS",
        "show_title": True,
        "show_legend": True,
    },
    ("D1", "delta_M"): {
        "ylim": (-3, 0.5),
        "ytick_step": 1.0,
        "ax_w_mm": 25, "ax_h_mm": 30,
        "left_mm": 12, "right_mm": 20, "bottom_mm": 12, "top_mm": 7,
        "ylabel": "ΔM score",
        "title": "D1: ΔM",
        "show_title": True,
        "show_legend": True,
    },
    ("D2", "delta_S"): {
        "ylim": (-0.3, 1.6),
        "ytick_step": 0.5,
        "ax_w_mm": 25, "ax_h_mm": 30,
        "left_mm": 12, "right_mm": 20, "bottom_mm": 12, "top_mm": 7,
        "ylabel": "ΔS score",
        "title": "D2: ΔS",
        "show_title": True,
        "show_legend": True,
    },
    ("D2", "delta_M"): {
        "ylim": (-3, 0.5),
        "ytick_step": 1.0,
        "ax_w_mm": 25, "ax_h_mm": 30,
        "left_mm": 12, "right_mm": 20, "bottom_mm": 12, "top_mm": 7,
        "ylabel": "ΔM score",
        "title": "D2: ΔM",
        "show_title": True,
        "show_legend": True,
    },
}

mpl.rcParams.update({
    "font.family": "Arial", "font.size": 7,
    "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.5,
})

for name, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name} h5ad file not found: {path}")
if BINNING_MODE not in {"WT_reference", "condition_specific"}:
    raise ValueError("BINNING_MODE must be WT_reference or condition_specific")
if not 1 <= MIN_BASELINE_VALID_BINS <= len(BASELINE_BINS):
    raise ValueError("Invalid MIN_BASELINE_VALID_BINS")
for celltype in CELLTYPES:
    for metric in ("delta_S", "delta_M"):
        cfg = PLOT_CFG[(celltype, metric)]
        if cfg["ax_w_mm"] <= 0 or cfg["ax_h_mm"] <= 0:
            raise ValueError(f"Invalid axis size: {(celltype, metric)}")
        if cfg["ylim"] is not None and cfg["ylim"][0] >= cfg["ylim"][1]:
            raise ValueError(f"Invalid ylim: {(celltype, metric)}")
print("Results:", OUT)
print("Binning mode:", BINNING_MODE)


In [ ]:
def prepare_obs(ad, target):
    required = [DC1_COL, S_COL, M_COL, SAMPLE_COL, CONDITION_COL]
    missing = [x for x in required if x not in ad.obs.columns]
    if missing:
        raise KeyError(f"{target}: missing obs columns {missing}")
    if ad.obs_names.has_duplicates:
        raise ValueError(f"{target}: duplicate cell IDs")
    df = pd.DataFrame({
        "cell_id": ad.obs_names.astype(str),
        "sample": ad.obs[SAMPLE_COL].astype(str).to_numpy(),
        "condition": ad.obs[CONDITION_COL].astype(str).replace(CONDITION_ALIASES).to_numpy(),
        "dc1": pd.to_numeric(ad.obs[DC1_COL], errors="coerce").to_numpy(float),
        "S_score": pd.to_numeric(ad.obs[S_COL], errors="coerce").to_numpy(float),
        "M_score": pd.to_numeric(ad.obs[M_COL], errors="coerce").to_numpy(float),
    })
    if REVERSE_DC1[target]:
        df["dc1"] *= -1
    df = df[df["condition"].isin(CONDITIONS)].copy()
    for c in CONDITIONS:
        if not (df["condition"] == c).any():
            print(f"[WARNING] {target}: no {c} cells; this group will be absent")
    both_scores = np.isfinite(df["S_score"]) & np.isfinite(df["M_score"])
    df["valid_input"] = np.isfinite(df["dc1"]) & both_scores
    return df


def compute_bin_edges(df, target):
    edges_by_condition = {}
    reference_condition = "WT" if BINNING_MODE == "WT_reference" else None
    for c in CONDITIONS:
        defining_condition = reference_condition or c
        dc1 = df.loc[(df["condition"] == defining_condition) & df["valid_input"], "dc1"].to_numpy(float)
        if dc1.size == 0:
            if defining_condition == "WT":
                raise ValueError(f"{target}: no valid WT cells to define the reference")
            continue
        edges = np.quantile(dc1, np.linspace(0, 1, N_BINS+1))
        if len(np.unique(edges)) < N_BINS+1:
            raise ValueError(f"{target}/{c}: too many tied DC1 values to form {N_BINS} distinct quantile bins")
        edges_by_condition[c] = edges
    rows = []
    for c, edges in edges_by_condition.items():
        for b in range(N_BINS):
            rows.append({"celltype": target, "condition": c, "reference_condition": reference_condition or c,
                         "bin": b+1, "dc1_left": edges[b], "dc1_right": edges[b+1], "binning_mode": BINNING_MODE})
    return edges_by_condition, pd.DataFrame(rows)


def assign_bins(df, edges_by_condition, target):
    df = df.copy()
    df["bin"] = -1
    df["outside_reference_range"] = False
    for c, edges in edges_by_condition.items():
        sel = (df["condition"] == c) & df["valid_input"]
        v = df.loc[sel, "dc1"].to_numpy(float)
        if len(v) == 0:
            continue
        b = np.searchsorted(edges[1:-1], v, side="right") + 1
        df.loc[sel, "bin"] = b
        df.loc[sel, "outside_reference_range"] = (v < edges[0]) | (v > edges[-1])
    outside = df.loc[df["valid_input"]].groupby(["condition","sample"], observed=True)["outside_reference_range"].agg(["sum", "count"]).reset_index()
    outside["fraction"] = outside["sum"] / outside["count"].clip(lower=1)
    outside.insert(0, "celltype", target)
    if BINNING_MODE == "WT_reference":
        print(f"{target}: cells outside the WT reference range are assigned to the nearest endpoint bin; see the group out-of-range fraction CSV")
    return df, outside


def summarize_per_animal(df, target, edges_by_condition):
    result = []
    qc = []
    valid = df[df["valid_input"] & (df["bin"] >= 1)]
    for condition in CONDITIONS:
        group = valid[valid["condition"] == condition]
        if group.empty:
            continue
        edges = edges_by_condition[condition]
        for sample in sorted(group["sample"].unique()):
            g = group[group["sample"] == sample]
            means_s = np.full(N_BINS, np.nan)
            means_m = np.full(N_BINS, np.nan)
            ns = np.zeros(N_BINS, dtype=int)
            for b in range(1, N_BINS+1):
                tmp = g[g["bin"] == b]
                ns[b-1] = len(tmp)
                if ns[b-1] >= MIN_CELLS_PER_BIN:
                    means_s[b-1] = tmp["S_score"].mean()
                    means_m[b-1] = tmp["M_score"].mean()
            baseline_ix = np.asarray(BASELINE_BINS, dtype=int)-1
            baseline_valid = np.isfinite(means_s[baseline_ix]) & np.isfinite(means_m[baseline_ix])
            n_base = int(baseline_valid.sum())
            ok = n_base >= MIN_BASELINE_VALID_BINS
            base_s = float(np.mean(means_s[baseline_ix][baseline_valid])) if ok else np.nan
            base_m = float(np.mean(means_m[baseline_ix][baseline_valid])) if ok else np.nan
            qc.append({"celltype": target,"condition":condition,"sample":sample,
                       "total_valid_cells":len(g), "valid_bins":int((ns >= MIN_CELLS_PER_BIN).sum()),
                       "valid_baseline_bins":n_base, "baseline_ok":ok})
            for j in range(N_BINS):
                s = means_s[j]
                m = means_m[j]
                delta_s = s-base_s if ok and np.isfinite(s) else np.nan
                delta_m = m-base_m if ok and np.isfinite(m) else np.nan
                result.append({
                    "celltype":target, "condition":condition,"sample":sample,"bin":j+1,
                    "dc1_left":float(edges[j]),"dc1_right":float(edges[j+1]),
                    "n_cells":int(ns[j]),"usable_bin":bool(ns[j] >= MIN_CELLS_PER_BIN),
                    "baseline_ok":bool(ok),"n_baseline_bins":n_base,
                    "raw_S":s,"raw_M":m,"raw_SminusM":s-m,
                    "baseline_S":base_s,"baseline_M":base_m,
                    "delta_S":delta_s,"delta_M":delta_m,
                    "delta_SminusM":delta_s-delta_m if np.isfinite(delta_s) and np.isfinite(delta_m) else np.nan,
                })
    return pd.DataFrame(result),pd.DataFrame(qc)


def summarize_conditions(profiles):
    cols = ["delta_S", "delta_M", "delta_SminusM"]
    rows = []
    for (celltype,cond,b), df in profiles.groupby(["celltype","condition","bin"], sort=False, observed=True):
        record = {"celltype":celltype,"condition":cond,"bin":int(b),"n_samples_total":df["sample"].nunique()}
        for col in cols:
            v = df[col].to_numpy(float)
            v = v[np.isfinite(v)]
            record[f"{col}_n"] = len(v)
            record[f"{col}_mean"] = float(np.mean(v)) if v.size else np.nan
            record[f"{col}_sd"] = float(np.std(v, ddof=1)) if v.size > 1 else np.nan
            record[f"{col}_sem"] = record[f"{col}_sd"] / np.sqrt(len(v)) if v.size > 1 else np.nan
        rows.append(record)
    return pd.DataFrame(rows)


In [ ]:
def _new_axes(target, metric):
    cfg = PLOT_CFG[(target, metric)]
    fw = cfg["left_mm"] + cfg["ax_w_mm"] + cfg["right_mm"]
    fh = cfg["bottom_mm"] + cfg["ax_h_mm"] + cfg["top_mm"]
    fig = plt.figure(figsize=(fw / 25.4, fh / 25.4))
    ax = fig.add_axes([
        cfg["left_mm"] / fw, cfg["bottom_mm"] / fh,
        cfg["ax_w_mm"] / fw, cfg["ax_h_mm"] / fh,
    ])
    ax.axhline(0, color="black", lw=0.5, ls="--", zorder=0)
    ax.set_xticks(np.arange(1, N_BINS + 1))
    ax.set_xlim(0.8, N_BINS + 0.2)
    ax.set_xlabel("DC1 bin (WT reference)" if BINNING_MODE == "WT_reference"
                  else "DC1 percentile bin (by condition)")
    ax.set_ylabel(cfg["ylabel"])
    if cfg.get("show_title", True):
        ax.set_title(cfg.get("title", ""), pad=3)
    if cfg["ylim"] is not None:
        ax.set_ylim(*cfg["ylim"])
    if cfg.get("ytick_step") is not None:
        ax.yaxis.set_major_locator(MultipleLocator(cfg["ytick_step"]))
    ax.tick_params(width=0.5, length=2, pad=1)
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    return fig, ax


def _line_and_sem(ax, subset, metric, condition, alpha=0.10):
    subset = subset.sort_values("bin")
    if subset.empty:
        return
    x = subset["bin"].to_numpy(int)
    y = subset[f"{metric}_mean"].to_numpy(float)
    sem = subset[f"{metric}_sem"].to_numpy(float)
    if SHOW_SEM:
        valid = np.isfinite(y) & np.isfinite(sem)
        if valid.any():
            ax.fill_between(x, y - sem, y + sem, where=valid,
                            interpolate=False, color=COLORS[condition],
                            alpha=alpha, linewidth=0)
    ax.plot(x, y, ls="-", color=COLORS[condition], lw=1,
            marker="o", markersize=1.4)


def save_single_delta_line(target, summary, metric):
    cfg = PLOT_CFG[(target, metric)]
    fig, ax = _new_axes(target, metric)
    for condition in CONDITIONS:
        dd = summary[(summary["celltype"] == target) &
                     (summary["condition"] == condition)]
        _line_and_sem(ax, dd, metric, condition)
    if cfg.get("show_legend", True):
        handles = [Line2D([0], [0], color=COLORS[c], lw=1, label=c)
                   for c in CONDITIONS]
        ax.legend(handles=handles, frameon=False, loc="upper left",
                  bbox_to_anchor=(1.02, 1.0), borderaxespad=0,
                  handlelength=1.8, handletextpad=0.5)

    path = OUT / f"{target}_{metric}_{BINNING_MODE}_mean_sem.pdf"
    try:

        fig.savefig(path, dpi=DPI, transparent=True)
    finally:
        plt.close(fig)
    return path


In [ ]:
all_profiles = []
all_summaries = []
all_qc = []
all_edges = []
all_outside = []
all_dist = []

for target in CELLTYPES:
    print(f"\n========== {target} ==========")
    ad = sc.read_h5ad(FILES[target], backed="r")
    try:
        df = prepare_obs(ad, target)
    finally:
        if ad.isbacked: ad.file.close()
        del ad
    edges, edge_table = compute_bin_edges(df, target)
    df, outside = assign_bins(df, edges, target)
    profiles, qc = summarize_per_animal(df, target, edges)
    if profiles.empty:
        print(f"[WARNING] {target}: no profiles to plot")
        continue
    summary = summarize_conditions(profiles)
    dist = df[df["valid_input"]].groupby(["condition","sample"],observed=True)["dc1"].agg(
        ["count","min","median","max"]).reset_index()
    dist.insert(0,"celltype",target)
    all_profiles.append(profiles)
    all_summaries.append(summary)
    all_qc.append(qc)
    all_edges.append(edge_table)
    all_outside.append(outside)
    all_dist.append(dist)

    print("Usable bins and baseline QC:")
    print(qc.to_string(index=False))
    for metric in ["delta_S", "delta_M", "delta_SminusM"]:
        print(metric, "range:", np.nanmin(profiles[metric].to_numpy()) if profiles[metric].notna().any() else "all missing",
              np.nanmax(profiles[metric].to_numpy()) if profiles[metric].notna().any() else "all missing")
    print("Saved:", save_single_delta_line(target, summary, "delta_S"))
    print("Saved:", save_single_delta_line(target, summary, "delta_M"))
    del df, profiles, summary, qc, edge_table, outside, dist
    gc.collect()

if not all_profiles:
    raise RuntimeError("No valid results. Check condition, sample, DC1, and score fields")

pd.concat(all_profiles,ignore_index=True).to_csv(OUT/f"01_sample_bin_score_profiles_{BINNING_MODE}.csv",index=False,encoding="utf-8-sig")
pd.concat(all_summaries,ignore_index=True).to_csv(OUT/f"02_condition_mean_sd_sem_{BINNING_MODE}.csv",index=False,encoding="utf-8-sig")
pd.concat(all_qc,ignore_index=True).to_csv(OUT/f"03_sample_baseline_bin_QC_{BINNING_MODE}.csv",index=False,encoding="utf-8-sig")
pd.concat(all_edges,ignore_index=True).to_csv(OUT/f"04_bin_edges_{BINNING_MODE}.csv",index=False,encoding="utf-8-sig")
pd.concat(all_outside,ignore_index=True).to_csv(OUT/f"05_cells_outside_reference_range_{BINNING_MODE}.csv",index=False,encoding="utf-8-sig")
pd.concat(all_dist,ignore_index=True).to_csv(OUT/f"06_dc1_range_by_sample_{BINNING_MODE}.csv",index=False,encoding="utf-8-sig")
print("\nDone. Results:", OUT)


## Panel E | DC1 gene heatmaps

Uses pooled-genotype DC1 quantile boundaries, WT gene ordering, and row Z scores. Expression falls back to `.X` if `norm_1e4` is absent. This differs from the explicit linear-expression requirement of the CV workflow below.


In [ ]:
# Imports and settings.
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
import logging, gc

DATA_DIR = Path(r"F:\BASSFISH_HD\Source_data")
FILES = {
    "D1": DATA_DIR / "d1_sm_data_final.h5ad",
    "D2": DATA_DIR / "d2_sm_data_final.h5ad",
}
OUT_ROOT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\sm") / "heatmap"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
CONDITIONS = ["WT", "Heter", "Homo"]
DC1_COL = "dc_component1"
SAMPLE_COL = "sample"
LAYER = "norm_1e4"
N_BINS = 10
ORDER_MODE = "peak_bin"
VMIN, VMAX, CMAP = -2.5, 2.5, "RdBu_r"
SHOW_GENE_NAMES = True
SHOW_COUNTS = True
DPI = 600
SHOW_COMBINED_FIGURE = True

PANEL_W, PANEL_H, GAP, GROUP_GAP = 24, 70, 2, 5
LEFT, RIGHT, BOTTOM, TOP, CB_W, CB_GAP = 13, 4, 17, 10, 3, 4

plt.rcParams.update({"font.family":"Arial", "font.size":7, "axes.labelsize":7,
                     "axes.titlesize":7, "xtick.labelsize":7, "ytick.labelsize":7,
                     "pdf.fonttype":42, "ps.fonttype":42})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)


USE_WT_GENE_QC = True
MANUAL_GENES = None
QC_SOURCE = Path(r"F:\BASSFISH_HD\Source_data\integrated_5_str_with_axis_xy.h5ad")
QC_MIN_WT_MEAN_NORM = 1.0
QC_MIN_WT_DETECTION = 0.02
QC_MIN_WT_SAMPLE_DETECTION = 0.01
QC_MIN_WT_SAMPLE_FRACTION = 0.50
QC_MIN_MATCH_RATE = 0.90
QC_MIN_BIN_CELLS = 5
QC_MIN_VALID_WT_BINS = 8


def filter_gene_mask(ad, target, output_dir, wt_bins=None):
    """Return the gene selection mask and export its QC audit."""
    all_genes = np.asarray(ad.var_names.astype(str))
    if len(set(all_genes)) != len(all_genes):
        raise ValueError(f'{target}: duplicate gene names prevent reliable filtering')
    selected = np.ones(len(all_genes), bool)
    if MANUAL_GENES is not None:
        requested = list(dict.fromkeys(str(g) for g in MANUAL_GENES))
        missing = sorted(set(requested) - set(all_genes))
        if missing:
            raise KeyError(f'{target}: selected genes are absent from var_names: {missing}')
        selected &= np.isin(all_genes, requested)

    audit = pd.DataFrame({'gene': all_genes, 'manual_pass': selected})
    if USE_WT_GENE_QC:
        if wt_bins is None:
            raise ValueError('WT expression QC requires the DC1 bins')
        if not QC_SOURCE.exists():
            raise FileNotFoundError(f'WT gene QC source not found: {QC_SOURCE}')
        source = sc.read_h5ad(QC_SOURCE)
        if 'norm_1e4' not in source.layers or 'counts' not in source.layers:
            raise KeyError('WT QC source requires layers["norm_1e4"] and layers["counts"]')
        if not ad.obs_names.is_unique or not source.obs_names.is_unique:
            raise ValueError('Duplicate cell IDs prevent unambiguous expression matching')
        if not source.var_names.is_unique:
            raise ValueError('QC source has duplicate gene names')
        target_wt = ad.obs['condition'].astype(str).to_numpy() == 'WT'
        target_wt &= np.isfinite(pd.to_numeric(ad.obs['dc_component1'], errors='coerce').to_numpy(float))
        ti = np.flatnonzero(target_wt)
        loc = source.obs_names.get_indexer(ad.obs_names[ti])
        found = loc >= 0
        matched_rate = float(found.mean()) if len(found) else 0.
        print(f'{target} WT expression-QC cell-ID match: {int(found.sum())}/{len(found)} = {matched_rate:.2%}')
        if matched_rate < QC_MIN_MATCH_RATE:
            raise ValueError(f'{target}: QC WT cell ID match rate {matched_rate:.2%} is below {QC_MIN_MATCH_RATE:.0%}')
        ti = ti[found]
        sj = loc[found]
        if 'sample' in source.obs:
            ss = source.obs.iloc[sj]['sample'].astype(str).to_numpy()
            ts = ad.obs.iloc[ti]['sample'].astype(str).to_numpy()
            bad = ss != ts
            if bad.any():
                raise ValueError(f'{target}: sample annotations differ for {int(bad.sum())} matched WT cells')
        if 'condition' in source.obs:
            scnd = source.obs.iloc[sj]['condition'].astype(str).to_numpy()
            if np.any(scnd != 'WT'):
                raise ValueError(f'{target}: some target WT cells are not labeled WT in the QC source')
        gidx = source.var_names.get_indexer(all_genes)
        present = gidx >= 0

        matched = gidx[present]
        nr = len(ti)
        norm = source.layers['norm_1e4'][sj,:][:,matched]
        counts = source.layers['counts'][sj,:][:,matched]
        if sparse.issparse(norm): norm = norm.toarray()
        if sparse.issparse(counts): counts = counts.toarray()
        norm = np.asarray(norm, dtype=float)
        counts = np.asarray(counts, dtype=float)
        norm_mean = np.nanmean(norm, axis=0) if nr else np.full(len(matched), np.nan)
        detected = np.mean(counts > 0, axis=0) if nr else np.zeros(len(matched))
        snames = ad.obs.iloc[ti]['sample'].astype(str).to_numpy()
        snames_unique = pd.unique(snames)
        per_sample = np.asarray([np.mean(counts[snames==s] > 0, axis=0) for s in snames_unique]) if len(snames_unique) else np.zeros((0,len(matched)))
        sample_frac = np.mean(per_sample >= QC_MIN_WT_SAMPLE_DETECTION, axis=0) if len(snames_unique) else np.zeros(len(matched))
        matched_bins = np.asarray(wt_bins)[ti]
        valid_bin_count = np.zeros(len(matched), int)
        for b in range(10):
            bsel = matched_bins == b
            if bsel.sum() >= QC_MIN_BIN_CELLS:

                valid_bin_count += np.any(counts[bsel] > 0, axis=0).astype(int)
        full_mean = np.full(len(all_genes), np.nan)
        full_detect = np.full(len(all_genes), np.nan)
        full_sample = np.full(len(all_genes), np.nan)
        full_bins = np.zeros(len(all_genes), int)
        full_mean[present] = norm_mean
        full_detect[present] = detected
        full_sample[present] = sample_frac
        full_bins[present] = valid_bin_count
        wt_pass = present & (full_mean >= QC_MIN_WT_MEAN_NORM) & (full_detect >= QC_MIN_WT_DETECTION) & (full_sample >= QC_MIN_WT_SAMPLE_FRACTION) & (full_bins >= QC_MIN_VALID_WT_BINS)
        selected &= wt_pass
        audit['present_in_QC_source'] = present
        audit['WT_mean_norm_1e4'] = full_mean
        audit['WT_detection_frac'] = full_detect
        audit['WT_fraction_samples_detect_ge_1pct'] = full_sample
        audit['WT_valid_gene_detected_bins'] = full_bins
        audit['wt_expression_qc_pass'] = wt_pass
        del norm, counts, source
    audit['pass_all_filters'] = selected
    output_dir.mkdir(parents=True, exist_ok=True)
    audit.to_csv(output_dir / f'{target}_optional_gene_filter_QC.csv', index=False)
    print(f'{target}: gene filter enabled={USE_WT_GENE_QC}, manual={MANUAL_GENES is not None}, pass={selected.sum()}/{len(selected)}')
    if not np.any(selected):
        raise ValueError(f'{target}: no genes passed filtering; check thresholds and the input source')
    return selected


In [ ]:
def mean_rows(X, idx):
    if len(idx) == 0:
        return np.full(X.shape[1], np.nan)
    z = X[idx]
    return np.asarray(z.mean(0)).ravel() if sparse.issparse(z) else np.nanmean(np.asarray(z), 0)

def quantile_bins(x, n=10):
    x = np.asarray(x, float)
    good = np.isfinite(x)
    if good.sum() == 0:
        raise ValueError("All DC1 values are NaN")
    edges = np.nanquantile(x[good], np.linspace(0, 1, n + 1))
    if len(np.unique(edges)) < n + 1:
        raise ValueError("Too many tied DC1 values to form 10 quantile bins reliably")
    bins = np.full(len(x), -1, int)
    bins[good] = np.clip(np.digitize(x[good], edges[1:-1]), 0, n-1)
    return bins, edges

def row_z(mat):
    mu = np.nanmean(mat, 1, keepdims=True)
    sd = np.nanstd(mat, 1, ddof=0, keepdims=True)
    return np.divide(mat - mu, sd, out=np.zeros_like(mat, float),
                     where=np.isfinite(sd) & (sd > 0))

def gene_order(wt):
    x = np.arange(wt.shape[1], dtype=float)
    peak = np.nanargmax(wt, 1)
    r = np.full(wt.shape[0], np.nan)
    for i, y in enumerate(wt):
        good = np.isfinite(y)
        if good.sum() >= 3 and np.nanstd(y[good]) > 0:
            r[i] = np.corrcoef(x[good], y[good])[0,1]
    order = (np.lexsort((np.nan_to_num(r), peak)) if ORDER_MODE == "peak_bin"
             else np.argsort(np.nan_to_num(r)))
    return order, peak, r


In [ ]:
def render_panels(target, panels, order, gene_names, filename, counts_map):
    """Arrange the precomputed heatmap matrices into panels."""
    if not panels:
        return

    positions = []
    x = LEFT
    prev_group = None
    for p in panels:
        group = p[1]
        if prev_group is not None and group != prev_group:
            x += GROUP_GAP - GAP
        positions.append(x)
        x += PANEL_W + GAP
        prev_group = group
    right = positions[-1] + PANEL_W
    fw = right + CB_GAP + CB_W + RIGHT
    fh = BOTTOM + PANEL_H + TOP
    fig = plt.figure(figsize=(fw / 25.4, fh / 25.4))
    im = None
    for i, ((name, group, z), xp) in enumerate(zip(panels, positions)):
        ax = fig.add_axes([xp/fw, BOTTOM/fh, PANEL_W/fw, PANEL_H/fh])
        im = ax.imshow(z[order], aspect="auto", interpolation="nearest",
                       cmap=CMAP, vmin=VMIN, vmax=VMAX, rasterized=False)
        ax.set_title(name, pad=2)
        ax.set_xticks(np.arange(N_BINS))
        ns = counts_map[(name, group)]
        labels = ([f"{b+1}\n{ns[b]}" for b in range(N_BINS)]
                  if SHOW_COUNTS else [str(b+1) for b in range(N_BINS)])
        ax.set_xticklabels(labels)
        ax.set_xlabel("DC1 bin")
        if i == 0 and SHOW_GENE_NAMES:
            ax.set_yticks(np.arange(len(gene_names)))
            ax.set_yticklabels(gene_names)
        else:
            ax.set_yticks([])
        ax.tick_params(width=.5, length=2, pad=1)
        for s in ax.spines.values():
            s.set_linewidth(.5)
    for i in range(1, len(panels)):
        if panels[i][1] != panels[i-1][1]:
            xpos = (positions[i-1] + PANEL_W + positions[i]) / 2
            fig.add_artist(plt.Line2D([xpos/fw, xpos/fw], [BOTTOM/fh, (BOTTOM+PANEL_H)/fh],
                                      transform=fig.transFigure, color="0.55", lw=.6, ls="--"))
    cax = fig.add_axes([(right+CB_GAP)/fw, BOTTOM/fh, CB_W/fw, PANEL_H/fh])
    cb = fig.colorbar(im, cax=cax)
    cb.set_label("Row z-score")
    cb.ax.tick_params(width=.5, length=2, pad=1)
    cb.outline.set_linewidth(.5)
    path = OUT_ROOT / filename
    try:
        fig.savefig(path, dpi=DPI, bbox_inches="tight", pad_inches=.02, transparent=True)
    finally:
        plt.close(fig)
    print("Saved:", path)


In [ ]:
def analyze_heatmaps(target):
    source_path = FILES[target]
    if not source_path.exists():
        raise FileNotFoundError(source_path)
    ad = sc.read_h5ad(source_path)
    print("\n", target, ad)
    for col in [DC1_COL, "condition", SAMPLE_COL]:
        if col not in ad.obs:
            raise KeyError(f"{target}: missing obs[{col!r}]")
    dc1 = pd.to_numeric(ad.obs[DC1_COL], errors="coerce").to_numpy(float)
    cond = ad.obs["condition"].astype(str).to_numpy()
    sample = ad.obs[SAMPLE_COL].astype(str).to_numpy()
    good = np.isfinite(dc1)
    X = ad.layers[LAYER] if LAYER in ad.layers else ad.X
    used_layer = LAYER if LAYER in ad.layers else "X"
    print(f"Expression source: {used_layer}")
    bins, edges = quantile_bins(dc1, N_BINS)
    selected = filter_gene_mask(ad, target, OUT_ROOT, wt_bins=bins)
    all_genes = np.asarray(ad.var_names.astype(str))
    genes = all_genes[selected]
    nvars = len(genes)


    pooled_raw = {}; pooled_z = {}; pooled_n = {}
    for c in CONDITIONS:
        mat = np.full((ad.n_vars, N_BINS), np.nan)
        ns = np.zeros(N_BINS, int)
        for b in range(N_BINS):
            idx = np.where(good & (cond == c) & (bins == b))[0]
            ns[b] = len(idx)
            if len(idx):
                mat[:,b] = mean_rows(X, idx)
        pooled_raw[c] = mat[selected]
        pooled_z[c] = row_z(pooled_raw[c])
        pooled_n[c] = ns
        print(target, c, "pooled bins:", ns.tolist())


    sample_info = []
    for c in CONDITIONS:
        for s in sorted(pd.unique(sample[good & (cond == c)])):
            sample_info.append((c,s))
    sample_z = {}; sample_n = {}; sample_qc = []
    for c,s in sample_info:
        mat = np.full((ad.n_vars, N_BINS), np.nan)
        ns = np.zeros(N_BINS, int)
        for b in range(N_BINS):

            idx = np.where(good & (sample == s) & (bins == b))[0]
            ns[b] = len(idx)
            if len(idx):
                mat[:,b] = mean_rows(X, idx)
            sample_qc.append({"condition":c,"sample":s,"bin":b+1,
                              "DC1_left":edges[b],"DC1_right":edges[b+1],"n_cells":ns[b]})
        sample_z[(c,s)] = row_z(mat[selected])
        sample_n[(c,s)] = ns
        print(target, c, s, "bins:", ns.tolist())


    order, peak, pearson_r = gene_order(pooled_z["WT"])
    sorted_names = genes[order]
    pd.DataFrame({"rank":np.arange(1,nvars+1),"gene":sorted_names,
                  "WT_peak_bin":peak[order]+1,"WT_pearson_r":pearson_r[order]}).to_csv(
        OUT_ROOT / f"{target}_DC1_quantile10_gene_order_shared.csv", index=False)

    rows = []
    for c in CONDITIONS:
        for i,g in enumerate(genes):
            record={"gene":g,"condition":c}
            record.update({f"bin_{b+1}":pooled_z[c][i,b] for b in range(N_BINS)})
            rows.append(record)
    pd.DataFrame(rows).to_csv(OUT_ROOT / f"{target}_DC1_quantile10_condition_profiles_row_zscore.csv", index=False)
    pd.DataFrame(sample_qc).to_csv(OUT_ROOT / f"{target}_DC1_quantile10_all_samples_bin_counts.csv", index=False)
    rows=[]
    for c,s in sample_info:
        mat = sample_z[(c,s)]
        for i,g in enumerate(genes):
            record = {"condition":c, "sample":s, "gene":g}
            record.update({f"bin_{b+1}":mat[i,b] for b in range(N_BINS)})
            rows.append(record)
    pd.DataFrame(rows).to_csv(OUT_ROOT / f"{target}_DC1_quantile10_all_samples_profiles_row_zscore.csv", index=False)

    count_df = pd.DataFrame({"bin":np.arange(1,N_BINS+1), "DC1_left":edges[:-1], "DC1_right":edges[1:]})
    for c in CONDITIONS:
        count_df[f"{c}_n_cells"] = pooled_n[c]
    count_df["all_n_cells"] = count_df[[f"{c}_n_cells" for c in CONDITIONS]].sum(axis=1)
    count_df.to_csv(OUT_ROOT / f"{target}_DC1_quantile10_bin_counts.csv", index=False)
    pd.DataFrame([{"target":target, "layer":used_layer, "n_input_genes":ad.n_vars,
                   "n_selected_genes":nvars, "n_cells":ad.n_obs,
                   "binning":"pooled quantiles all conditions (original)",
                   "normalization":"per gene row z (original)",
                   "order":"WT pooled original peak_bin", "USE_WT_GENE_QC":USE_WT_GENE_QC,
                   "MANUAL_GENES":str(MANUAL_GENES)}]).to_csv(
        OUT_ROOT / f"{target}_DC1_quantile10_parameters.csv", index=False)

    condition_panels = [(c,c,pooled_z[c]) for c in CONDITIONS]
    condition_count_map = {(c,c): pooled_n[c] for c in CONDITIONS}
    per_sample_panels = [(f"{s}\n{c}",c,sample_z[(c,s)]) for c,s in sample_info]

    per_sample_counts = {(f"{s}\n{c}",c):sample_n[(c,s)] for c,s in sample_info}
    render_panels(target, condition_panels, order, sorted_names,
                  f"{target}_DC1_quantile10_WT_Heter_Homo_gene_heatmap.pdf", condition_count_map)
    render_panels(target, per_sample_panels, order, sorted_names,
                  f"{target}_DC1_quantile10_all_samples_gene_heatmap.pdf", per_sample_counts)
    if SHOW_COMBINED_FIGURE:

        combo_cond = [(f"Pooled {c}", "Pooled", pooled_z[c]) for c in CONDITIONS]
        combo_counts = {(f"Pooled {c}","Pooled"):pooled_n[c] for c in CONDITIONS}
        render_panels(target, combo_cond + per_sample_panels, order, sorted_names,
                      f"{target}_DC1_quantile10_condition_plus_per_sample.pdf",
                      {**combo_counts, **per_sample_counts})
    del ad, X
    gc.collect()
    return {"target":target,"input_genes":len(all_genes),"selected_genes":nvars,"samples":len(sample_info),"used_layer":used_layer}

summary=[]
for target in ["D1", "D2"]:
    summary.append(analyze_heatmaps(target))
pd.DataFrame(summary).to_csv(OUT_ROOT / "_D1_D2_heatmap_summary.csv", index=False)
print("Done:", OUT_ROOT)


## Panel F | Spatial CV along DC1

Uses linear expression with cell/gene ID matching, genotype-specific DC1 quantiles, ten valid bins per animal/gene, paired-gene Wilcoxon tests, and BH correction.


In [ ]:
# Imports and settings.
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc, matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import wilcoxon


ROOT=Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\sm")
OUT_ROOT=ROOT/"RDR_CV_verified_linear_expression"
OUT=OUT_ROOT/"relative_dynamic_range"
OUT.mkdir(parents=True,exist_ok=True)
FILES={
    "D1":Path(r"F:\BASSFISH_HD\Source_data\d1_sm_data_final.h5ad"),
    "D2":Path(r"F:\BASSFISH_HD\Source_data\d2_sm_data_final.h5ad"),
}


CONDITIONS=["WT","Heter","Homo"]
condition_cmap={"WT":"#4cd964","Heter":"#ff9500","Homo":"#ff3b30"}

DC1_COL="dc_component1"; COND_COL="condition"; SAMPLE_COL="sample"
LAYER="norm_1e4"
N_BINS=10
MIN_VALID_BINS=10


VARIATION_METRIC="relative_dynamic_range"

SHOW_PAIRED_LINES=False
DPI=600

AXES_WIDTH_MM=28; AXES_HEIGHT_MM=50
LEFT_MARGIN_MM=12; RIGHT_MARGIN_MM=7; BOTTOM_MARGIN_MM=12; TOP_MARGIN_MM=10

plt.rcParams.update({
    "font.family":"Arial","font.size":7,"axes.labelsize":7,"axes.titlesize":7,
    "xtick.labelsize":7,"ytick.labelsize":7,"legend.fontsize":7,
    "pdf.fonttype":42,"ps.fonttype":42
})


def mean_rows(X,rows):
    if len(rows)==0: return np.full(X.shape[1],np.nan)
    z=X[rows]
    return np.asarray(z.mean(axis=0)).ravel() if sparse.issparse(z) else np.mean(np.asarray(z),axis=0)

def calc_variation(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<MIN_VALID_BINS: return np.nan
    mean=np.nanmean(x); dr=np.nanmax(x)-np.nanmin(x)
    if VARIATION_METRIC=="relative_dynamic_range":
        return dr/mean if mean>0 else np.nan
    if VARIATION_METRIC=="cv":
        return np.nanstd(x,ddof=0)/mean if mean>0 else np.nan
    if VARIATION_METRIC=="dynamic_range":
        return dr
    raise ValueError("VARIATION_METRIC must be relative_dynamic_range / cv / dynamic_range")

def bh_fdr(p):
    p=np.asarray(p,float); out=np.full(len(p),np.nan)
    ok=np.isfinite(p)
    if ok.sum()==0: return out
    pv=p[ok]; order=np.argsort(pv); ranked=pv[order]
    q=ranked*len(ranked)/np.arange(1,len(ranked)+1)
    q=np.minimum.accumulate(q[::-1])[::-1]; q=np.clip(q,0,1)
    tmp=np.empty(len(pv)); tmp[order]=q; out[ok]=tmp
    return out

def p_label(p):
    if not np.isfinite(p): return "NA"
    if p<0.0001: return "****"
    if p<0.001: return "***"
    if p<0.01: return "**"
    if p<0.05: return "*"
    return "ns"

def add_sig(ax,x1,x2,y,p):
    yr=ax.get_ylim()[1]-ax.get_ylim()[0]; h=yr*0.025
    ax.plot([x1,x1,x2,x2],[y,y+h,y+h,y],color="black",lw=.5,clip_on=False)
    ax.text((x1+x2)/2,y+h,p_label(p),ha="center",va="bottom",fontsize=7)


USE_WT_GENE_QC = True
MANUAL_GENES = None
QC_SOURCE = Path(r"F:\BASSFISH_HD\Source_data\integrated_5_str_with_axis_xy.h5ad")
QC_MIN_WT_MEAN_NORM = 1.0
QC_MIN_WT_DETECTION = 0.02
QC_MIN_WT_SAMPLE_DETECTION = 0.01
QC_MIN_WT_SAMPLE_FRACTION = 0.50
QC_MIN_MATCH_RATE = 0.90
QC_MIN_BIN_CELLS = 5
QC_MIN_VALID_WT_BINS = 8


def filter_gene_mask(ad, target, output_dir, wt_bins=None):
    """Return the gene selection mask and export its QC audit."""
    all_genes = np.asarray(ad.var_names.astype(str))
    if len(set(all_genes)) != len(all_genes):
        raise ValueError(f'{target}: duplicate gene names prevent reliable filtering')
    selected = np.ones(len(all_genes), bool)
    if MANUAL_GENES is not None:
        requested = list(dict.fromkeys(str(g) for g in MANUAL_GENES))
        missing = sorted(set(requested) - set(all_genes))
        if missing:
            raise KeyError(f'{target}: selected genes are absent from var_names: {missing}')
        selected &= np.isin(all_genes, requested)

    audit = pd.DataFrame({'gene': all_genes, 'manual_pass': selected})
    if USE_WT_GENE_QC:
        if wt_bins is None:
            raise ValueError('WT expression QC requires the DC1 bins')
        if not QC_SOURCE.exists():
            raise FileNotFoundError(f'WT gene QC source not found: {QC_SOURCE}')
        source = sc.read_h5ad(QC_SOURCE)
        if 'norm_1e4' not in source.layers or 'counts' not in source.layers:
            raise KeyError('WT QC source requires layers["norm_1e4"] and layers["counts"]')
        if not ad.obs_names.is_unique or not source.obs_names.is_unique:
            raise ValueError('Duplicate cell IDs prevent unambiguous expression matching')
        if not source.var_names.is_unique:
            raise ValueError('QC source has duplicate gene names')
        target_wt = ad.obs['condition'].astype(str).to_numpy() == 'WT'
        target_wt &= np.isfinite(pd.to_numeric(ad.obs['dc_component1'], errors='coerce').to_numpy(float))
        ti = np.flatnonzero(target_wt)
        loc = source.obs_names.get_indexer(ad.obs_names[ti])
        found = loc >= 0
        matched_rate = float(found.mean()) if len(found) else 0.
        print(f'{target} WT expression-QC cell-ID match: {int(found.sum())}/{len(found)} = {matched_rate:.2%}')
        if matched_rate < QC_MIN_MATCH_RATE:
            raise ValueError(f'{target}: QC WT cell ID match rate {matched_rate:.2%} is below {QC_MIN_MATCH_RATE:.0%}')
        ti = ti[found]
        sj = loc[found]
        if 'sample' in source.obs:
            ss = source.obs.iloc[sj]['sample'].astype(str).to_numpy()
            ts = ad.obs.iloc[ti]['sample'].astype(str).to_numpy()
            bad = ss != ts
            if bad.any():
                raise ValueError(f'{target}: sample annotations differ for {int(bad.sum())} matched WT cells')
        if 'condition' in source.obs:
            scnd = source.obs.iloc[sj]['condition'].astype(str).to_numpy()
            if np.any(scnd != 'WT'):
                raise ValueError(f'{target}: some target WT cells are not labeled WT in the QC source')
        gidx = source.var_names.get_indexer(all_genes)
        present = gidx >= 0

        matched = gidx[present]
        nr = len(ti)
        norm = source.layers['norm_1e4'][sj,:][:,matched]
        counts = source.layers['counts'][sj,:][:,matched]
        if sparse.issparse(norm): norm = norm.toarray()
        if sparse.issparse(counts): counts = counts.toarray()
        norm = np.asarray(norm, dtype=float)
        counts = np.asarray(counts, dtype=float)
        norm_mean = np.nanmean(norm, axis=0) if nr else np.full(len(matched), np.nan)
        detected = np.mean(counts > 0, axis=0) if nr else np.zeros(len(matched))
        snames = ad.obs.iloc[ti]['sample'].astype(str).to_numpy()
        snames_unique = pd.unique(snames)
        per_sample = np.asarray([np.mean(counts[snames==s] > 0, axis=0) for s in snames_unique]) if len(snames_unique) else np.zeros((0,len(matched)))
        sample_frac = np.mean(per_sample >= QC_MIN_WT_SAMPLE_DETECTION, axis=0) if len(snames_unique) else np.zeros(len(matched))
        matched_bins = np.asarray(wt_bins)[ti]
        valid_bin_count = np.zeros(len(matched), int)
        for b in range(10):
            bsel = matched_bins == b
            if bsel.sum() >= QC_MIN_BIN_CELLS:

                valid_bin_count += np.any(counts[bsel] > 0, axis=0).astype(int)
        full_mean = np.full(len(all_genes), np.nan)
        full_detect = np.full(len(all_genes), np.nan)
        full_sample = np.full(len(all_genes), np.nan)
        full_bins = np.zeros(len(all_genes), int)
        full_mean[present] = norm_mean
        full_detect[present] = detected
        full_sample[present] = sample_frac
        full_bins[present] = valid_bin_count
        wt_pass = present & (full_mean >= QC_MIN_WT_MEAN_NORM) & (full_detect >= QC_MIN_WT_DETECTION) & (full_sample >= QC_MIN_WT_SAMPLE_FRACTION) & (full_bins >= QC_MIN_VALID_WT_BINS)
        selected &= wt_pass
        audit['present_in_QC_source'] = present
        audit['WT_mean_norm_1e4'] = full_mean
        audit['WT_detection_frac'] = full_detect
        audit['WT_fraction_samples_detect_ge_1pct'] = full_sample
        audit['WT_valid_gene_detected_bins'] = full_bins
        audit['wt_expression_qc_pass'] = wt_pass
        del norm, counts, source
    audit['pass_all_filters'] = selected
    output_dir.mkdir(parents=True, exist_ok=True)
    audit.to_csv(output_dir / f'{target}_optional_gene_filter_QC.csv', index=False)
    print(f'{target}: gene filter enabled={USE_WT_GENE_QC}, manual={MANUAL_GENES is not None}, pass={selected.sum()}/{len(selected)}')
    if not np.any(selected):
        raise ValueError(f'{target}: no genes passed filtering; check thresholds and the input source')
    return selected


EXPRESSION_SOURCE = QC_SOURCE
EXPRESSION_LAYER = "norm_1e4"
MIN_EXPRESSION_MATCH_RATE = 0.99
LINEAR_RTOL = 1e-4
LINEAR_ATOL = 1e-6
_EXPRESSION_SOURCE_CACHE = None

def get_expression_source():
    global _EXPRESSION_SOURCE_CACHE
    if _EXPRESSION_SOURCE_CACHE is None:
        z = sc.read_h5ad(EXPRESSION_SOURCE)
        if not z.obs_names.is_unique or not z.var_names.is_unique:
            raise ValueError("Source object contains duplicate cell IDs or gene names")
        for layer in [EXPRESSION_LAYER, "counts"]:
            if layer not in z.layers:
                raise KeyError(f"Source object is missing {layer}; fallback to .X/raw is disabled")
        _EXPRESSION_SOURCE_CACHE = z
    return _EXPRESSION_SOURCE_CACHE

def load_verified_expression(ad, target):
    source = get_expression_source()
    if not ad.obs_names.is_unique or not ad.var_names.is_unique:
        raise ValueError(f"{target}: target object contains duplicate cell IDs or gene names")
    cell_idx = source.obs_names.get_indexer(ad.obs_names)
    gene_idx = source.var_names.get_indexer(ad.var_names)
    if np.any(gene_idx < 0):
        raise ValueError(f"{target}: missing genes {list(ad.var_names[gene_idx < 0])}")
    matched = cell_idx >= 0
    if not len(matched) or matched.mean() < MIN_EXPRESSION_MATCH_RATE:
        raise ValueError(f"{target}: cell match rate is below {MIN_EXPRESSION_MATCH_RATE:.1%}")
    audit = pd.DataFrame({"cell_id": ad.obs_names.astype(str),
                          "sample": ad.obs[SAMPLE_COL].astype(str).to_numpy(),
                          "condition": ad.obs[COND_COL].astype(str).to_numpy(),
                          "matched_expression": matched})
    audit.to_csv(OUT / f"{target}_expression_cell_match_audit.csv", index=False)
    audit.loc[~matched].to_csv(OUT / f"{target}_excluded_unmatched_cells.csv", index=False)
    summary = audit.groupby(["condition", "sample"], observed=True).agg(
        n_cells=("cell_id", "size"), n_matched=("matched_expression", "sum")).reset_index()
    summary["n_excluded"] = summary.n_cells - summary.n_matched
    summary.to_csv(OUT / f"{target}_expression_match_by_sample.csv", index=False)
    aligned_obs = source.obs.iloc[cell_idx[matched]]
    for col in [SAMPLE_COL, COND_COL]:
        if col not in source.obs:
            raise KeyError(f"Source object is missing obs column {col} required for identity checks")
        a = ad.obs.loc[matched, col].astype(str).replace({"HD":"Heter"}).to_numpy()
        b = aligned_obs[col].astype(str).replace({"HD":"Heter"}).to_numpy()
        if np.any(a != b):
            raise ValueError(f"{target}: {col} differs in {int(np.sum(a != b))} cells")
    row = cell_idx[matched]
    def take(layer):
        a = source.layers[layer][row, :][:, gene_idx]
        return a.toarray().astype(float) if sparse.issparse(a) else np.asarray(a, dtype=float)
    counts = take("counts")
    norm = take(EXPRESSION_LAYER)
    if not np.isfinite(counts).all() or not np.isfinite(norm).all():
        raise ValueError(f"{target}: counts or norm contains nonfinite values")
    if np.any(counts < 0) or np.any(norm < 0):
        raise ValueError(f"{target}: counts or norm contains negative values")
    zero_mismatch = (counts == 0) != (norm == 0)
    csum = counts.sum(axis=1)
    nsum = norm.sum(axis=1)
    scale = np.divide(nsum, csum, out=np.zeros_like(nsum), where=csum>0)
    expected = counts * scale[:, None]
    bad_rows = ~np.all(np.isclose(norm, expected, rtol=LINEAR_RTOL, atol=LINEAR_ATOL), axis=1)
    bad_rows |= np.any(zero_mismatch, axis=1)
    bad_rows |= (csum > 0) & (scale <= 0)
    pd.DataFrame([{"celltype":target, "n_cells":ad.n_obs,
                   "n_matched":int(matched.sum()), "n_unmatched":int((~matched).sum()),
                   "n_genes":ad.n_vars, "expression_source":str(EXPRESSION_SOURCE),
                   "expression_layer":EXPRESSION_LAYER,
                   "n_failed_linear_rows":int(bad_rows.sum()),
                   "n_zero_mismatches":int(zero_mismatch.sum()),
                   "min_expression":float(norm.min()), "max_expression":float(norm.max())}]
                 ).to_csv(OUT / f"{target}_linear_expression_validation.csv", index=False)
    if bad_rows.any():
        failed = audit.loc[matched].iloc[np.flatnonzero(bad_rows)]
        failed.to_csv(OUT / f"{target}_failed_linear_validation_cells.csv", index=False)
        raise ValueError(f"{target}: {int(bad_rows.sum())} cells do not satisfy per-cell linear scaling of counts. Check the source layer.")

    expression = np.zeros((ad.n_obs, ad.n_vars), dtype=float)
    expression[matched] = norm
    print(f"{target}: linear-expression check passed; matched {matched.sum()}/{ad.n_obs}, excluded {(~matched).sum()}")
    print(summary.to_string(index=False))
    return expression, f"source.layers[{EXPRESSION_LAYER}]", matched


In [ ]:
def analyze(target):
    adata=sc.read_h5ad(FILES[target]); print("\n",target,adata)
    for col in [DC1_COL,COND_COL,SAMPLE_COL]:
        if col not in adata.obs: raise KeyError(f"{target}: obs is missing {col}")


    if USE_WT_GENE_QC or MANUAL_GENES is not None:
        _dc=pd.to_numeric(adata.obs[DC1_COL],errors="coerce").to_numpy(float)
        _cond=adata.obs[COND_COL].astype(str).to_numpy()
        _wt_good=np.isfinite(_dc)&(_cond=="WT")
        _edges=np.nanquantile(_dc[_wt_good],np.linspace(0,1,N_BINS+1))
        _wt_bins=np.full(adata.n_obs,-1,int)
        _wt_bins[_wt_good]=np.clip(np.digitize(_dc[_wt_good],_edges[1:-1]),0,N_BINS-1)
        _mask=filter_gene_mask(adata,target,OUT,wt_bins=_wt_bins)
        if not np.all(_mask):
            adata=adata[:,_mask].copy()
    X,used_layer,expression_matched=load_verified_expression(adata,target)
    genes=adata.var_names.astype(str).to_numpy()
    dc1=pd.to_numeric(adata.obs[DC1_COL],errors="coerce").to_numpy(float)
    cond=adata.obs[COND_COL].astype(str).to_numpy()
    sample=adata.obs[SAMPLE_COL].astype(str).to_numpy()
    reference_valid=np.isfinite(dc1)
    valid=reference_valid & expression_matched


    bins=np.full(adata.n_obs,-1,int); edge_rows=[]
    for c in CONDITIONS:
        m=reference_valid&(cond==c)
        if m.sum()==0: raise ValueError(f"{target}: {c} has no valid DC1 values")
        edges=np.nanquantile(dc1[m],np.linspace(0,1,N_BINS+1))
        if len(np.unique(edges))<N_BINS+1:
            raise ValueError(f"{target} {c}: too many tied DC1 values to form {N_BINS} quantile bins reliably")
        bins[m]=np.clip(np.digitize(dc1[m],edges[1:-1]),0,N_BINS-1)
        for b in range(N_BINS):
            edge_rows.append({"celltype":target,"condition":c,"bin":b+1,
                              "DC1_left":edges[b],"DC1_right":edges[b+1],
                              "n_cells":int((m&(bins==b)).sum()),
                              "n_expression_matched_cells":int((m&(bins==b)&expression_matched).sum())})

    edges_df=pd.DataFrame(edge_rows)
    edges_df.to_csv(OUT/f"{target}_DC1_gene_variation_quantile_bin_QC.csv",index=False)


    records=[]
    for c in CONDITIONS:
        samples_c=sorted(pd.unique(sample[valid&(cond==c)]))
        print(c,"samples:",samples_c)

        for s in samples_c:
            sm=valid&(cond==c)&(sample==s)
            mat=np.full((adata.n_vars,N_BINS),np.nan)

            for b in range(N_BINS):
                rows=np.where(sm&(bins==b))[0]
                mat[:,b]=mean_rows(X,rows)

            for i,g in enumerate(genes):
                records.append({
                    "celltype":target,"gene":g,"condition":c,"sample":s,
                    "variation":calc_variation(mat[i,:]),
                    "mean_expression_across_bins":np.nanmean(mat[i,:]),
                    "n_valid_bins":int(np.isfinite(mat[i,:]).sum())
                })

    sample_var=pd.DataFrame(records)
    if VARIATION_METRIC == "cv":
        finite_cv=sample_var["variation"].dropna()
        if (finite_cv > np.sqrt(N_BINS-1) + 1e-6).any():
            raise AssertionError("Nonnegative 10-bin CV exceeds the theoretical upper bound; check the input")
    sample_csv=OUT/f"{target}_DC1_all_gene_sample_variation.csv"
    sample_var.to_csv(sample_csv,index=False)


    gene_var=sample_var.groupby(["celltype","gene","condition"],observed=True)["variation"].agg(
        ["mean","std","count"]).reset_index()
    gene_var["sem"]=gene_var["std"]/np.sqrt(gene_var["count"])
    gene_var=gene_var.rename(columns={"mean":"variation"})
    gene_csv=OUT/f"{target}_DC1_all_gene_condition_variation_one_point_per_gene.csv"
    gene_var.to_csv(gene_csv,index=False)

    print("\nGenes per condition:")
    print(gene_var.groupby("condition")["variation"].apply(lambda x:x.notna().sum()))


    wide=gene_var.pivot(index="gene",columns="condition",values="variation")
    stats=[]
    for a,b in [("WT","Heter"),("WT","Homo"),("Heter","Homo")]:
        pair=wide[[a,b]].dropna()
        if len(pair)>=2:
            try: stat,p=wilcoxon(pair[a],pair[b])
            except ValueError: stat,p=np.nan,np.nan
        else: stat,p=np.nan,np.nan
        stats.append({"celltype":target,"group1":a,"group2":b,
                      "n_genes":len(pair),"statistic":stat,"p_value":p})

    stats_df=pd.DataFrame(stats)
    stats_df["p_adj_BH"]=bh_fdr(stats_df["p_value"].to_numpy())
    stats_df["significance"]=stats_df["p_adj_BH"].map(p_label)
    stats_csv=OUT/f"{target}_DC1_all_gene_paired_wilcoxon.csv"
    stats_df.to_csv(stats_csv,index=False)
    print("\nPaired Wilcoxon:")
    print(stats_df)


    fig_w=LEFT_MARGIN_MM+AXES_WIDTH_MM+RIGHT_MARGIN_MM
    fig_h=BOTTOM_MARGIN_MM+AXES_HEIGHT_MM+TOP_MARGIN_MM
    fig=plt.figure(figsize=(fig_w/25.4,fig_h/25.4))
    ax=fig.add_axes([LEFT_MARGIN_MM/fig_w,BOTTOM_MARGIN_MM/fig_h,
                     AXES_WIDTH_MM/fig_w,AXES_HEIGHT_MM/fig_h])

    positions=np.arange(1,4)
    box_data=[gene_var.loc[gene_var["condition"].eq(c),"variation"].dropna().to_numpy()
              for c in CONDITIONS]

    bp=ax.boxplot(box_data,positions=positions,widths=.55,patch_artist=True,showfliers=False,
                  medianprops=dict(color="black",linewidth=.7),
                  boxprops=dict(color="black",linewidth=.6),
                  whiskerprops=dict(color="black",linewidth=.6),
                  capprops=dict(color="black",linewidth=.6))

    for patch,c in zip(bp["boxes"],CONDITIONS):
        patch.set_facecolor(condition_cmap[c]); patch.set_alpha(.25)

    if SHOW_PAIRED_LINES:
        common=wide.dropna(subset=CONDITIONS)
        for _,row in common.iterrows():
            ax.plot(positions,[row[c] for c in CONDITIONS],color=".82",lw=.35,zorder=1)

    rng=np.random.default_rng(1)
    for x0,c in zip(positions,CONDITIONS):
        vals=gene_var.loc[gene_var["condition"].eq(c),"variation"].dropna().to_numpy()
        jitter=rng.normal(0,.07,size=len(vals))
        ax.scatter(np.full(len(vals),x0)+jitter,vals,s=2,color=condition_cmap[c],
                   alpha=.55,edgecolors="none",zorder=3)

    ax.set_xticks(positions); ax.set_xticklabels(CONDITIONS)
    ylabel={"relative_dynamic_range":"Relative spatial variation\n(max − min) / mean",
            "cv":"Spatial variation (CV)",
            "dynamic_range":"Spatial dynamic range"}[VARIATION_METRIC]
    ax.set_ylabel(ylabel)
    ax.set_title(f"{target} | DC1 | n={adata.n_vars} genes",pad=3)
    ax.tick_params(width=.5,length=2,pad=1)
    for sp in ax.spines.values(): sp.set_linewidth(.5)


    finite=np.concatenate([x[np.isfinite(x)] for x in box_data if len(x)])
    ymin,ymax=np.nanmin(finite),np.nanmax(finite); span=max(ymax-ymin,1e-6)
    ax.set_yscale("linear")
    ax.set_ylim(0, 1)
    y0=ymax+.05*span; step=.075*span
    for k,(a,b) in enumerate([("WT","Heter"),("Heter","Homo"),("WT","Homo")]):
        row=stats_df[(stats_df.group1.eq(a))&(stats_df.group2.eq(b))]
        if len(row)==0:
            row=stats_df[(stats_df.group1.eq(b))&(stats_df.group2.eq(a))]
        p=float(row["p_adj_BH"].iloc[0]) if len(row) else np.nan
        x1=CONDITIONS.index(a)+1; x2=CONDITIONS.index(b)+1
        add_sig(ax,x1,x2,y0+k*step,p)

    pdf=OUT/f"{target}_DC1_all_gene_{VARIATION_METRIC}_scatter_boxplot.pdf"
    png=OUT/f"{target}_DC1_all_gene_{VARIATION_METRIC}_scatter_boxplot.png"
    fig.savefig(pdf,dpi=DPI,bbox_inches="tight",pad_inches=.02,transparent=True)
    fig.savefig(png,dpi=DPI,bbox_inches="tight",pad_inches=.02)
    plt.close(fig)

    print("\nSaved:")
    print(pdf); print(sample_csv); print(gene_csv); print(stats_csv)


In [ ]:
METRICS_TO_RUN = ["cv"]

for metric in METRICS_TO_RUN:
    VARIATION_METRIC = metric
    OUT = OUT_ROOT / metric
    OUT.mkdir(parents=True, exist_ok=True)
    print(f"\n{'='*60}\nMETRIC = {metric}, OUT = {OUT}\n{'='*60}")
    for target in ["D1", "D2"]:
        analyze(target)
print("\nVerified-linear-expression results saved:", OUT_ROOT)


In [ ]:
all_medians=[]; all_changes=[]; all_tests=[]
for metric in METRICS_TO_RUN:
    folder=OUT_ROOT/metric
    for target in ["D1", "D2"]:
        gene=pd.read_csv(folder/f"{target}_DC1_all_gene_condition_variation_one_point_per_gene.csv")
        stat=pd.read_csv(folder/f"{target}_DC1_all_gene_paired_wilcoxon.csv")
        vals={c:gene.loc[gene.condition.eq(c), "variation"].dropna().to_numpy(float)
              for c in CONDITIONS}
        med={c:float(np.median(v)) if len(v) else np.nan for c,v in vals.items()}
        for c in CONDITIONS:
            all_medians.append(dict(metric=metric, celltype=target, condition=c,
                                    n_valid_genes=len(vals[c]), median=med[c]))
        for c in ["Heter", "Homo"]:
            row=stat.loc[stat.group1.eq("WT") & stat.group2.eq(c)].iloc[0]
            decrease=100*(med["WT"]-med[c])/med["WT"] if med["WT"]>0 else np.nan
            all_changes.append(dict(metric=metric, celltype=target, comparison=f"{c}_vs_WT",
                WT_n_valid_genes=len(vals["WT"]), HD_n_valid_genes=len(vals[c]),
                WT_median_all_valid=med["WT"], HD_median_all_valid=med[c],
                decrease_percent=decrease, n_paired_genes=int(row.n_genes),
                p_value=row.p_value, q_BH=row.p_adj_BH))
        stat["metric"]=metric;all_tests.append(stat)
medians=pd.DataFrame(all_medians)
changes=pd.DataFrame(all_changes)
tests=pd.concat(all_tests,ignore_index=True)
medians.to_csv(OUT_ROOT/"ALL_condition_medians.csv",index=False)
changes.to_csv(OUT_ROOT/"ALL_WT_relative_decrease_and_statistics.csv",index=False)
tests.to_csv(OUT_ROOT/"ALL_pairwise_statistics.csv",index=False)
print("Median across all valid genes in each group:")
print(medians.to_string(index=False))
print("Percentage decrease relative to WT (negative values indicate increases); n_paired_genes is the number of genes in the paired test:")
print(changes.to_string(index=False))
print("Output directory:",OUT_ROOT)
